# 049 — Dog vs Cat: the first complete CNN

Three things in this lesson are new, and the architecture is not one of them:
images arrive through a **generator**, the model **overfits visibly**, and
**99.4%** of its parameters sit in the dense head.

TensorFlow is not needed: the parameter counts and the memory arithmetic are
computed from the architecture.

| Part | What we check |
|---|---|
| A | the layer-by-layer count, totalling **14,848,193** |
| B | 0.6% conv against **99.4%** dense |
| C | global average pooling shrinks the model **125x** |
| D | why a generator: **14.6 GiB** held at once against **24 MiB** |
| E | reading overfitting from the gap between two curves |

## Part A — The architecture, counted

In [ ]:
BLOCKS = [32, 64, 128]      # filters per convolution block
DENSE = [128, 64]

h = w = 256
c = 3
rows, total = [("Input", f"{h}x{w}x{c}", 0)], 0

for filters in BLOCKS:
    params = (3 * 3 * c + 1) * filters + 4 * filters   # conv weights + batch norm
    h, w = h - 2, w - 2                               # 3x3 filter, valid padding
    rows.append((f"Conv2D({filters}) + BN", f"{h}x{w}x{filters}", params))
    total += params
    h, w = h // 2, w // 2                             # 2x2 max pool
    rows.append(("MaxPooling2D(2x2)", f"{h}x{w}x{filters}", 0))
    c = filters

flat = h * w * c
rows.append(("Flatten", f"{flat:,}", 0))
for units in DENSE:
    rows.append((f"Dense({units})", str(units), flat * units + units))
    total += flat * units + units
    flat = units
rows.append(("Dense(1, sigmoid)", "1", flat + 1))
total += flat + 1

print(f"{'layer':<26}{'output':>16}{'params':>13}")
for name, shape, p in rows:
    print(f"{name:<26}{shape:>16}{p:>13,}")
print(f"{'':<26}{'TOTAL':>16}{total:>13,}")
assert total == 14_848_193

## Part B — Where the parameters live

In [ ]:
conv = sum(p for n, _, p in rows if n.startswith("Conv"))
dense = sum(p for n, _, p in rows if n.startswith("Dense"))
print(f"conv  {conv:>12,}  ({conv / total:.1%})")
print(f"dense {dense:>12,}  ({dense / total:.1%})")
assert round(dense / total * 100, 1) == 99.4
assert round(conv / total * 100, 1) == 0.6

first_dense = 30 * 30 * 128 * 128 + 128
print(f"the first Dense alone: {first_dense:,} - it eats a "
      f"{30 * 30 * 128:,}-value vector")
assert first_dense == 14_745_728
print("the layer facing the large flattened vector dominates - again")

## Part C — What global average pooling would save

Instead of flattening 30x30x128 into 115,200 values, average each of the 128
feature maps to a single number.

In [ ]:
gap_head = 128 * 128 + 128            # Dense(128) on a 128-value input
shrunk = total - first_dense + gap_head
print(f"flatten head : {total:>12,}")
print(f"GAP head     : {shrunk:>12,}")
print(f"ratio        : {total / shrunk:.0f}x smaller")
assert gap_head == 16_512
assert shrunk == 118_977 and round(total / shrunk) == 125
print("the model that does the seeing is unchanged; only the head shrinks")

## Part D — Why the images arrive through a generator

In [ ]:
N, PIX, BATCH = 20_000, 256 * 256 * 3, 32
full = N * PIX * 4                     # float32 after normalisation
batch = BATCH * PIX * 4
print(f"all {N:,} images as float32 : {full / 2**30:.1f} GiB")
print(f"one batch of {BATCH}          : {batch / 2**20:.0f} MiB")
print(f"ratio                      : {full / batch:,.0f}x less RAM at once")
assert round(full / 2**30, 1) == 14.6
assert round(batch / 2**20) == 24
assert round(full / batch) == 625

## Part E — Diagnosing overfitting from the gap

Neither curve on its own tells you anything. The gap does. These are the shapes
you are looking for, not measured numbers.

In [ ]:
# a worked reading of three (train, validation) accuracy pairs
readings = [(0.72, 0.70, "underfitting - both low, still improving"),
            (0.91, 0.89, "healthy - small gap, both rising"),
            (0.99, 0.82, "overfitting - train still climbing, validation fell")]
for train, val, verdict in readings:
    print(f"train {train:.0%}  val {val:.0%}  gap {train - val:+.0%}   {verdict}")

worst = max(readings, key=lambda r: r[0] - r[1])
assert worst[2].startswith("overfitting")
print()
print("order to try fixes in: more data -> augmentation -> dropout/L2 ->")
print("a smaller model -> early stopping. More data first, always.")

## What this notebook established

- The dog-vs-cat CNN has **14,848,193** parameters, of which **99.4%** are
  dense and **0.6%** convolutional.
- One layer — the first `Dense`, facing a 115,200-value flatten — holds
  **14,745,728** of them.
- Replacing `Flatten` with `GlobalAveragePooling2D` leaves **118,977**
  parameters, **125x** smaller, without touching the convolutional stack.
- Holding all 20,000 images as float32 needs **14.6 GiB**; a batch of 32 needs
  **24 MiB**, which is why a generator is not an optimisation but a
  requirement.

## Exercises

1. Add a fourth convolution block of 256 filters. Does the dense share go up or
   down? Explain the direction before you compute it.
2. At what input size would the flatten produce fewer than 10,000 values?
3. Work out the parameter count if the two hidden dense layers were swapped
   (64 then 128). Why does the order barely matter?